In [1]:
import torch
from torch.nn import functional as F
from kerops.ops.linear.linear_bias_relu_linear_backward import LinBReLULinBackward, autotune_lin_bn_relu_lin_backward, generate_inputs_lin_bn_relu_lin_backward
from kerops.ops.assets import ASSETS_ROOT

In [3]:
autotune_lin_bn_relu_lin_backward(ASSETS_ROOT / 'LinBReLULinBackward.toml', n_jobs_precompile=4)

Problem sizes:   0%|          | 0/3 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/36 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/36 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/24 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/24 [00:00<?, ?it/s]

Precompiling:   0%|          | 0/24 [00:00<?, ?it/s]

Benchmark configs:   0%|          | 0/24 [00:00<?, ?it/s]

In [37]:
channels = 64

x, grad, weight_up, weight_down, bias = generate_inputs_lin_bn_relu_lin_backward({'in_channels': channels})

In [39]:
%%timeit -r 10 -n 10
LinBReLULinBackward(x, grad, weight_up, weight_down, bias)
torch.cuda.synchronize()

5.3 ms ± 79.6 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [33]:
from torch import nn


model = nn.Sequential(
    nn.Conv3d(channels, channels * 2, kernel_size=1, bias=True),
    nn.ReLU(inplace=True),
    nn.Conv3d(2 * channels, channels, kernel_size=1, bias=False)
).cuda()
model.half()

Sequential(
  (0): Conv3d(64, 128, kernel_size=(1, 1, 1), stride=(1, 1, 1))
  (1): ReLU(inplace=True)
  (2): Conv3d(128, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
)

In [34]:
with torch.amp.autocast('cuda'):
    output = model(x)

In [36]:
%%timeit -r 10 -n 10
model.zero_grad()

with torch.amp.autocast('cuda'), torch.inference_mode():
    output.backward(grad, retain_graph=True)
torch.cuda.synchronize()

3.29 ms ± 73 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)
